In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType, IntegerType
import time


In [2]:
spark = SparkSession.builder.appName("case-study").getOrCreate()
customers=spark.read \
    .option("header", True) \
    .csv("data/customers.csv")

order_items=spark.read \
    .option("header", True) \
    .csv("data/order_items.csv")
orders=spark.read \
    .option("header", True) \
    .csv("data/orders.csv")
products=spark.read \
    .option("header", True) \
    .csv("data/products.csv")
returns=spark.read \
    .option("header", True) \
    .csv("data/returns.csv")

customers.createOrReplaceTempView("c")
order_items.createOrReplaceTempView("oi")
orders.createOrReplaceTempView("o")
products.createOrReplaceTempView("p")
returns.createOrReplaceTempView("r")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 13:20:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print("total no. of customer", customers.count())
print("total no. of orders", orders.count())
print("total no. of products", products.count())
print("total no. of returns", returns.count())

total no. of customer 100000


total no. of orders 1000000
total no. of products 50000
total no. of returns 100000


In [4]:
sql1=spark.sql("""
    select category,
    sum(unit_cost)as total_sales
    from p
    group by category
""")
sql1.show()

sql1.write.mode("overwrite").csv("output/sql1", header=True)

+--------------+------------------+
|      category|       total_sales|
+--------------+------------------+
|Home & Kitchen| 2901364.330000004|
|        Sports| 2853163.040000003|
|   Electronics|2864604.7399999946|
|      Clothing| 2841424.610000002|
|         Books|2853871.8500000075|
|        Beauty|2919388.7500000037|
|          Toys|2851913.1100000013|
+--------------+------------------+



In [5]:
sql2=spark.sql("""
    select c.customer_name,
        sum(oi.selling_price*oi.quantity) as total_purchase_amount
    from
        c join 
        o on c.customer_id=o.customer_id
        join oi on o.order_id=oi.order_id
        join p on oi.product_id=p.product_id
    group by c.customer_name
    order by total_purchase_amount desc
    limit 10
""")
sql2.show()

sql2.write.mode("overwrite").csv("output/sql2", header=True)

26/06/15 13:21:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/06/15 13:21:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 13:21:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

+--------------+---------------------+
| customer_name|total_purchase_amount|
+--------------+---------------------+
|Customer_93094|   181569.68000000005|
|Customer_64560|   169060.39999999997|
|Customer_23289|             161573.8|
|Customer_52275|   153364.78999999998|
|Customer_61218|            153067.55|
|Customer_52034|            152680.05|
|Customer_40442|   151037.32000000004|
|Customer_60528|            148691.95|
|Customer_84830|            148363.84|
|Customer_82593|            148281.04|
+--------------+---------------------+



26/06/15 13:21:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 13:21:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [7]:
sql3 = spark.sql("""
    select 
        year(to_date(o.order_date, 'yyyy-MM-dd')) as year,
        month(to_date(o.order_date, 'yyyy-MM-dd')) as month_num,
        date_format(to_date(o.order_date, 'yyyy-MM-dd'), 'MMM') as month_name,
        sum(oi.selling_price * oi.quantity) as total_sales
    from o
    join oi on o.order_id = oi.order_id
    where year(to_date(o.order_date, 'yyyy-MM-dd')) = (
        select year(max(to_date(order_date, 'yyyy-MM-dd'))) FROM o
    )
    group by year, month_num, month_name
    order by month_num
""")

sql3.show()

sql3.write.mode("overwrite").csv("output/sql3", header=True)

+----+---------+----------+--------------------+
|year|month_num|month_name|         total_sales|
+----+---------+----------+--------------------+
|2024|        1|       Jan| 4.445777757600014E8|
|2024|        2|       Feb|4.1536614419999766E8|
|2024|        3|       Mar| 4.436282454099968E8|
|2024|        4|       Apr|4.2782097433999556E8|
|2024|        5|       May|4.4481061894999766E8|
|2024|        6|       Jun| 4.317051540600035E8|
|2024|        7|       Jul| 4.436705191200028E8|
|2024|        8|       Aug| 4.410951770200006E8|
|2024|        9|       Sep|4.3107152608000004E8|
|2024|       10|       Oct| 4.413637893100021E8|
|2024|       11|       Nov|4.3362336404000014E8|
|2024|       12|       Dec| 4.427129083499984E8|
+----+---------+----------+--------------------+



In [ ]:
sql4=spark.sql("""
    select p.category,
""")

In [ ]:
spark.stop()